# Task 3: RAG Pipeline Construction and Evaluation

- Objective

-Build an end-to-end Retrieval-Augmented Generation (RAG) pipeline using the pre-built FAISS vector store and evaluate its effectiveness via qualitative analysis. This notebook consumes the production modules implemented in src/ and utils/ and demonstrates correct wiring, execution, and evaluation per rubric.

In [ ]:
# Task 3: RAG Pipeline Construction and Evaluation

from typing import List, Dict, Any
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import sys
from pathlib import Path

import faiss
import pandas as pd
import pickle

# Ensure project root is on sys.path so local packages import in notebooks
_p = Path.cwd()
for _ancestor in [_p] + list(_p.parents):
    if (_ancestor / "src").exists() and (_ancestor / "utils").exists():
        sys.path.insert(0, str(_ancestor))
        break

from utils.paths import VECTOR_STORE_DIR
from src.retriever import Retriever, dynamic_k
from src.generator import AnswerGenerator
from src.rag_pipeline import RAGPipeline
from src.evaluation import evaluate_responses

print("VECTOR_STORE_DIR:", VECTOR_STORE_DIR)

## Load Vector Store

In [ ]:
# --- Load Vector Store ---
index_path = os.path.join(VECTOR_STORE_DIR, "index.faiss")
metadata_path = os.path.join(VECTOR_STORE_DIR, "metadata.pkl")

index = faiss.read_index(index_path)

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("FAISS vectors:", index.ntotal)
print("Metadata items:", len(metadata))
print("Metadata sample keys:", metadata[0].keys())

## Build retriever + generator + pipeline

In [ ]:
# --- Build retriever + generator + pipeline ---
retriever = Retriever(
    index=index,
    metadata=metadata,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

generator = AnswerGenerator(
    model_name="google/flan-t5-base",   # upgraded from small
    device=-1,                         # CPU
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
    generate_kwargs={
        # These are generally helpful; if your transformers version rejects them,
        # remove them and rerun (core pipeline still works).
        "repetition_penalty": 1.1,
        "no_repeat_ngram_size": 3,
    }
)

# Default k lowered; dynamic_k used for aggregate questions below
rag = RAGPipeline(retriever=retriever, generator=generator,
                  k=3, max_context_chars=900)

##  Single question test- Sanity Check

In [ ]:
# --- Single question test (Sanity Check) ---
res = rag.answer(
    "Do complaints mention discrimination? What situations are described?",
    k=dynamic_k(
        "Do complaints mention discrimination? What situations are described?", rag.k),
    return_prompt=True
)

print(res.answer)

print("\n--- Prompt (first 1200 chars) ---")
print((res.prompt or "")[:1200])

print("\n--- Sources (preview) ---")
for s in res.sources[:2]:
    print(s.get("complaint_id"), s.get("product"),
          s.get("score"), s.get("score_type"))
    print((s.get("text") or "")[:300], "\n")

## Qualitative evaluation run (5–10 questions)

In [ ]:
# --- Qualitative evaluation run (5–10 questions) ---
questions = [
    "What are the most common issues customers complain about regarding credit card charges?",
    "Do complaints mention discrimination? What situations are described?",
    "Are there recurring issues with loan payment posting delays or misapplied payments?",
    "What are customers complaining about regarding overdraft or unexpected fees?",
    "Do users report problems closing accounts or cancelling services?",
    "Are there complaints about credit reporting errors or incorrect delinquency reporting?",
    "What issues do customers report with customer service responsiveness?",
    "Do complaints mention fraud or unauthorized transactions?",
]

rag_results = []
for q in questions:
    kq = dynamic_k(q, rag.k)
    rag_results.append(rag.answer(q, k=kq))

results_for_eval: List[Dict[str, Any]] = [
    {"question": r.question, "answer": r.answer, "sources": r.sources}
    for r in rag_results
]

df = evaluate_responses(results_for_eval)

df_eval = df.copy()
df_eval["qualitative_feedback"] = ""
display(df_eval.head())

## Export Markdown table for your report

In [ ]:
# --- Export Markdown table for your report ---
print(df.to_markdown(index=False))

df.to_csv("task3_rag_qualitative_evaluation.csv", index=False)
print("Saved task3_rag_qualitative_evaluation.csv")

df_loaded = pd.read_csv("task3_rag_qualitative_evaluation.csv")
print("Loaded", len(df_loaded), "rows")
print(df_loaded.to_markdown(index=False))

## Save CSV for manual scoring/comments

In [ ]:
df.to_csv(r"D:\Python\Week 7\Intelligent-Complaint-Analysis\data\processed\task3_rag_qualitative_evaluation.csv", index=False)

In [ ]:
# Load and preview the saved qualitative evaluation results
import pandas as pd
df_loaded = pd.read_csv("task3_rag_qualitative_evaluation.csv")
print('Loaded', len(df_loaded), 'rows from task3_rag_qualitative_evaluation.csv')
print(df_loaded.to_markdown(index=False))

## Evaluation Analysis

